In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# Load the laptop buying dataset
# Empty cells and common missing-value words will be treated as NaN
df = pd.read_csv(
    "/content/laptop_buying_data.csv",
    na_values=["", " ", "NA", "N/A", "null", "NULL", "None", "?"]
)

df.head()

In [ ]:
# Basic inspection before building the model
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and missing values:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Remove exact duplicate rows
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

In [ ]:
# Look at the numerical columns
df.describe()

In [ ]:
# Check the target distribution
print(df["Buys Laptop"].value_counts(dropna=False))

df["Buys Laptop"].value_counts().plot(
    kind="bar",
    title="Laptop Buying Distribution"
)

plt.xlabel("Buys Laptop")
plt.ylabel("Number of Records")
plt.show()

In [ ]:
# Separate input features and target
X = df.drop("Buys Laptop", axis=1)
y = df["Buys Laptop"]

numeric_features = ["Age", "Credit Score"]
categorical_features = ["Income", "Student", "Location"]

In [ ]:
# Split first so test data does not influence preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
# Fill missing numerical values with the median
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Fill missing categorical values with the most frequent value
# and convert categories into numbers
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
# Apply the correct preprocessing to each type of column
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Preprocess the data, then train the Random Forest
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=42
    ))
])

# n_estimators = number of decision trees
# max_depth = maximum depth of each tree

In [ ]:
# Train the complete pipeline
model.fit(X_train, y_train)

print("Model trained successfully.")

In [ ]:
# Predict the test data
y_pred = model.predict(X_test)

print("Predicted:", y_pred)
print("Actual:   ", y_test.values)

In [ ]:
# Check classification performance
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 2))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print(cm)

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

In [ ]:
# Check performance across 5 different folds
cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation scores:", cv_scores)
print("Mean CV accuracy:", round(cv_scores.mean(), 2))

In [ ]:
# Get feature names after one-hot encoding
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

# Get importance values from the Random Forest
importances = model.named_steps["classifier"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

feature_importance

In [ ]:
# Visualize the most important features
top_features = feature_importance.head(10).sort_values("Importance")

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Feature Importances")
plt.show()

In [ ]:
# Test the model with a new person
new_person = pd.DataFrame({
    "Age": [26],
    "Income": ["Low"],
    "Student": ["No"],
    "Credit Score": [680],
    "Location": ["Urban"]
})

prediction = model.predict(new_person)[0]
probability = model.predict_proba(new_person).max()

print("Prediction:", prediction)
print("Confidence:", round(probability, 2))